In [6]:
import pandas as pd
import numpy as np
import glob

In [7]:
files = glob.glob("*.csv")

In [8]:
df_list = [pd.read_csv(file) for file in files]

In [9]:
df = pd.concat(df_list,ignore_index= True)

In [10]:
df = df.drop_duplicates(subset='ride_id')

In [11]:
df['started_at'] = pd.to_datetime(df['started_at'], format= 'mixed')

In [12]:
df['ended_at'] = pd.to_datetime(df['ended_at'], format= 'mixed')

In [13]:
df['date'] = df['started_at'].dt.floor('D')

In [ ]:
df['month'] = df['started_at'].dt.month_name()

In [15]:
df['day_of_week'] = df['started_at'].dt.day_name()

In [16]:
df['ride_duration_min'] = ((df['ended_at'] - df['started_at']).dt.total_seconds() / 60).round(2)


In [17]:
df = df[
    (df['ride_duration_min'] >= 2) &
    (df['ride_duration_min'] < 120)
]

In [18]:
df[['start_station_id','end_station_id','start_station_name','end_station_name' ]] = df[['start_station_id','end_station_id','start_station_name','end_station_name' ]].fillna('Unknown')

In [13]:
df['start_lat'].describe()

count    5.852186e+06
mean     4.190219e+01
std      4.473256e-02
min      4.164000e+01
25%      4.188096e+01
50%      4.189738e+01
75%      4.193000e+01
max      4.207000e+01
Name: start_lat, dtype: float64

In [23]:
df_grouped = df.groupby(['month', 'member_casual'])['ride_id'].count().reset_index()

In [24]:
print(df_grouped)

        month member_casual  ride_id
0       April        casual   123972
1       April        member   269548
2      August        casual   299051
3      August        member   420742
4    December        casual    36469
5    December        member   132974
6    February        casual    44915
7    February        member   168089
8     January        casual    22990
9     January        member   112531
10       July        casual   300522
11       July        member   412954
12       June        casual   281674
13       June        member   393950
14      March        casual    78234
15      March        member   208556
16        May        casual   216333
17        May        member   361503
18   November        casual    88578
19   November        member   232006
20    October        casual   204974
21    October        member   385219
22  September        casual   325286
23  September        member   454735


In [28]:
df_pivot = df_grouped.pivot_table(values = 'ride_id', index = 'month', columns = 'member_casual').reset_index()

In [29]:
print(df_pivot)

member_casual      month    casual    member
0                  April  123972.0  269548.0
1                 August  299051.0  420742.0
2               December   36469.0  132974.0
3               February   44915.0  168089.0
4                January   22990.0  112531.0
5                   July  300522.0  412954.0
6                   June  281674.0  393950.0
7                  March   78234.0  208556.0
8                    May  216333.0  361503.0
9               November   88578.0  232006.0
10               October  204974.0  385219.0
11             September  325286.0  454735.0


In [32]:
df_pivot['pct_diff'] = (df_pivot['member'] - df_pivot['casual']) / df_pivot['casual'] * 100

In [34]:
df_pivot['ratio'] = df_pivot['member'] / df_pivot['casual']

In [35]:
print(df_pivot)

member_casual      month    casual    member    pct_diff     ratio
0                  April  123972.0  269548.0  117.426516  2.174265
1                 August  299051.0  420742.0   40.692390  1.406924
2               December   36469.0  132974.0  264.622008  3.646220
3               February   44915.0  168089.0  274.238005  3.742380
4                January   22990.0  112531.0  389.478034  4.894780
5                   July  300522.0  412954.0   37.412236  1.374122
6                   June  281674.0  393950.0   39.860264  1.398603
7                  March   78234.0  208556.0  166.579748  2.665797
8                    May  216333.0  361503.0   67.104880  1.671049
9               November   88578.0  232006.0  161.922825  2.619228
10               October  204974.0  385219.0   87.935543  1.879355
11             September  325286.0  454735.0   39.795442  1.397954


In [36]:
sample_df = df.sample(n=2000, random_state=42) 

In [37]:
sample_df.to_csv("sample cleaned Data", index=False)

In [38]:
df.to_csv("cleaned_data.csv", index=False, decimal= '.')

In [39]:
df_pivot.to_csv("pivot table", index= False)